In [ ]:
# import os
# print("커널이 살아있습니다!")
# print(f"현재 경로: {os.getcwd()}")

In [4]:
%load_ext autotime

time: 54.8 μs (started: 2026-05-11 15:22:02 +00:00)


# 1. 라이브러리 임포트, 모델 로드, vLLM 엔진 초기화

In [5]:
print("1️⃣ 환경 설정 및 라이브러리 임포트 시작...")
import os
import subprocess
import time
# import requests
print("기타 라이브러리 임포트 완료")

print("OpenAI 클라이언트 임포트 시작")
# # vllm 엔진을 직접 로드하는 대신 서버와 통신할 클라이언트를 임포트함.
try:
    from openai import OpenAI
    print("- OpenAI 클라이언트 임포트 완료")
except ImportError:
    print("- OpenAI 라이브러리 설치 중...")
    os.system('pip install openai')
    from openai import OpenAI
    print("- OpenAI 클라이언트 임포트 완료")

print("모든 라이브러리 임포트 완료")
# ===========================================

print("✅ 환경 설정 완료")

1️⃣ 환경 설정 및 라이브러리 임포트 시작...
기타 라이브러리 임포트 완료
OpenAI 클라이언트 임포트 시작
- OpenAI 클라이언트 임포트 완료
모든 라이브러리 임포트 완료
✅ 환경 설정 완료
time: 742 μs (started: 2026-05-11 15:22:04 +00:00)


In [19]:
print("2️⃣ vLLM API 서버 백그라운드 기동...")

# 터미널에서 실행할 명령어 구성 (FP8 및 비디오 처리 최적화)
PYTHON_EXEC = "/workspace/envs/trimenders_v1/bin/python"
MODEL_PATH = "/workspace/models/Qwen3.6-35B-A3B-FP8"
LOG_FILE_PATH = "vllm_server.log"


# 추가됨: --allowed-local-media-path /workspace (로컬 파일 읽기 권한 부여)
vllm_cmd = f"""
{PYTHON_EXEC} -m vllm.entrypoints.openai.api_server \
    --model {MODEL_PATH} \
    --trust-remote-code \
    --dtype auto \
    --quantization fp8 \
    --tensor-parallel-size 2 \
    --max-model-len 16384 \
    --gpu-memory-utilization 0.90 \
    --reasoning-parser qwen3 \
    --limit-mm-per-prompt '{{"video": 1}}' \
    --media-io-kwargs '{{"video": {{"num_frames": -1}}}}' \
    --allowed-local-media-path /workspace \
    --port 8000
"""

# 1. 서버 백그라운드 실행
log_file = open(LOG_FILE_PATH, "w")
server_process = subprocess.Popen(vllm_cmd, shell=True, stdout=log_file, stderr=subprocess.STDOUT)

print("⏳ 서버 로딩 중... (실시간 로그 출력)")
print("-" * 60)

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# =====================================================================
# 2. 질문하신 대기 루프의 '업그레이드' 버전 (로그 출력 + 생존 감시 + API 체크)
# =====================================================================
with open(LOG_FILE_PATH, "r") as read_log:
    while True:
        # (1) 새롭게 추가된 로그를 읽어서 주피터 화면에 출력합니다.
        new_lines = read_log.readlines()
        for line in new_lines:
            print(line.strip())
        
        # (2) 프로세스가 예기치 않게(OOM 등) 종료되었는지 먼저 감시합니다.
        if server_process.poll() is not None:
            print("-" * 60)
            print(f"❌ 에러: vLLM 서버 프로세스가 종료되었습니다! (종료 코드: {server_process.returncode})")
            print("위쪽 로그에서 'Out of Memory' 등의 에러 원인을 확인하세요.")
            break

        # (3) 질문하신 부분: API가 응답하는지 찔러봅니다.
        try:
            client.models.list()
            print("-" * 60)
            print("✅ vLLM 서버 기동 완료! (포트 8000)")
            break # 성공했으므로 무한 루프 탈출!
        except Exception:
            # 아직 서버가 준비 안 됨. 
            # 실시간 로그를 빨리빨리 보기 위해 10초 대신 2초만 쉬고 다시 루프를 돕니다.
            time.sleep(2)

2️⃣ vLLM API 서버 백그라운드 기동...
⏳ 서버 로딩 중... (실시간 로그 출력)
------------------------------------------------------------
(APIServer pid=5311) INFO 05-11 15:56:24 [utils.py:299]
(APIServer pid=5311) INFO 05-11 15:56:24 [utils.py:299]        █     █     █▄   ▄█
(APIServer pid=5311) INFO 05-11 15:56:24 [utils.py:299]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.19.1
(APIServer pid=5311) INFO 05-11 15:56:24 [utils.py:299]   █▄█▀ █     █     █     █  model   /workspace/models/Qwen3.6-35B-A3B-FP8
(APIServer pid=5311) INFO 05-11 15:56:24 [utils.py:299]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=5311) INFO 05-11 15:56:24 [utils.py:299]
(APIServer pid=5311) INFO 05-11 15:56:24 [utils.py:233] non-default args: {'model': '/workspace/models/Qwen3.6-35B-A3B-FP8', 'trust_remote_code': True, 'allowed_local_media_path': '/workspace', 'max_model_len': 16384, 'quantization': 'fp8', 'reasoning_parser': 'qwen3', 'tensor_parallel_size': 2, 'limit_mm_per_prompt': {'video': 1}, 'media_io_kwargs': {'video': {'num_frames': -

In [10]:
# !apt-get update && apt-get install -y ffmpeg

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]                
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:6 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1294 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7004 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:9 http://archive.ubuntu.com/ubuntu jammy/main amd64 Packages [1792 kB]     
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3915 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa

In [32]:
ORIGIN_VIDEO = "/workspace/data/video/ai_hub_senior_anomaly/Validation/video/dementia_daily/M_I_004/DDA_In_MI004_00009.mp4"
TEMP_CLIP = "/workspace/members/kim/0511_temp_input_daily.mp4"

# START_TIME = 0      # 시작 시점 (초)
# DURATION = 20         # 추출할 길이 (초)
FPS = 5.0             # Qwen3.6이 분석할 초당 프레임 (가공이 아닌 API 요청 시 사용)

# print(f"3️⃣ 원본 영상에서 분석 구간({START_TIME}s ~ {START_TIME + DURATION}s) 추출 및 최적화 중...")
print("원본 영상에서 추출 및 최적화 중")

# -c copy를 사용하여 인코딩 없이 잘라냄.
# cmd = [
#     "ffmpeg", "-y", 
#     "-ss", str(START_TIME), 
#     "-t", str(DURATION), 
#     "-i", ORIGIN_VIDEO, 
#     "-c", "copy",      # 인코딩 없이 그대로 복사
#     "-loglevel", "error",
#     TEMP_CLIP
# ]


# 해상도를 720p로 낮추고 5FPS로 변환하여 AI가 소화하기 좋게 만듭니다.
cmd = [
    "ffmpeg", "-y", 
    # "-ss", str(START_TIME), 
    # "-t", str(DURATION), 
    "-i", ORIGIN_VIDEO, 
    "-vf", "scale=-2:720,fps=5", # 핵심: 720p 해상도 + 초당 5프레임으로 압축
    "-c:v", "libx264", 
    "-preset", "ultrafast",      # 초고속 인코딩 설정
    "-loglevel", "error",
    TEMP_CLIP
]

try:
    subprocess.run(cmd, check=True)
    file_size = os.path.getsize(TEMP_CLIP) / (1024 * 1024)
    print(f"✅ AI 최적화 클립 생성 완료: {TEMP_CLIP} ({file_size:.2f} MB)")
except subprocess.CalledProcessError as e:
    print(f"❌ 클립 생성 실패: {e}")

원본 영상에서 추출 및 최적화 중
✅ AI 최적화 클립 생성 완료: /workspace/members/kim/0511_temp_input_daily.mp4 (4.33 MB)
time: 8.23 s (started: 2026-05-11 16:26:33 +00:00)


In [42]:
# 프롬프트 구성

# System Prompt
system_prompt = """
You are an objective video analysis AI. Your output must ONLY consist of the [Activity Log] section.

[Strict Constraints]
1. Factual Description Only: Describe physical movements, velocity, and balance. If the center of gravity shifts faster than a controlled sitting motion, specify it as a 'loss of balance' or 'sudden collapse'.
2. No Summary/Status: Do not provide "Current Status", "Summary of Concerns", or "Movement Quality" sections.
3. Describe the direction of the fall based on which body part makes contact with the floor first (e.g., knees, chest, side, or back). Do not assume the direction unless the movement trajectory is clearly visible.
4. No Inference: Describe the fall as an "uncontrolled rapid descent" if it lacks bracing motions, but do not guess the medical cause.
5. Output Format: Strictly follow the format below without any additional text.
"""

# User Prompt
user_prompt = f"""
Analyze the provided video clip. 
Provide a factual description of the movements for each 10-second interval.

### [Activity Log]
- Interval 1 : 
- Interval 2 :
"""

time: 415 μs (started: 2026-05-11 16:40:35 +00:00)


In [43]:
start_time = time.time()

USE_THINKING = True  # False: 단순 답변, True: 사고 과정 포함

print("4️⃣ Qwen3.6-35B (Thinking Mode) 비디오 분석 시작...")
# 2. OpenAI API 호출
# start_time_inference = time.time()

extra_params = {"mm_processor_kwargs": {"fps": FPS, "do_sample_frames": True}}
if USE_THINKING:
    extra_params["chat_template_kwargs"] = {"enable_thinking": True}

response = client.chat.completions.create(
    model=MODEL_PATH,
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": f"file://{TEMP_CLIP}"}},
                {"type": "text", "text": user_prompt}
            ]
        }
    ],
    temperature=0.7 if USE_THINKING else 0.7, # 생각할 땐 창의적으로, 아닐 땐 일관적으로
    max_tokens=1024,
    extra_body=extra_params
)
end_time = time.time()

# ... 위쪽 API 호출 코드는 그대로 두시고, end_time = time.time() 아래부터 교체하세요 ...

print("\n" + "="*25 + " DEBUGGING ANALYSIS " + "="*25)
print(f"⏱️ 추론 소요 시간: {end_time - start_time:.2f}초")
print("-" * 67)

# 1. 응답 객체를 딕셔너리로 강제 변환
try:
    full_response = response.model_dump()
    message_dict = full_response['choices'][0]['message']
    finish_reason = full_response['choices'][0].get('finish_reason')
    
    print(f"🏁 종료 사유(Finish Reason): {finish_reason}")
    
    # 2. 텍스트 추출 시도
    content = message_dict.get('content', "")
    reasoning = message_dict.get('reasoning_content', "")
    
    if content and content.strip():
        print(f"📝 [최종 답변]:\n{content.strip()}")
    elif reasoning and reasoning.strip():
        print(f"🧠 [사고 과정에서 발견]:\n{reasoning.strip()}")
    else:
        # 3. 아무것도 없다면 JSON 전체 구조 출력 (정체 파악의 핵심)
        print("⚠️ 일반적인 텍스트 필드가 비어있습니다. 전체 응답 구조를 확인합니다:")
        import json
        print(json.dumps(full_response, indent=2, ensure_ascii=False))

except Exception as e:
    print(f"❌ 결과 해석 중 에러 발생: {e}")
    print(f"원본 응답 객체: {response}")

print("=" * 67)

4️⃣ Qwen3.6-35B (Thinking Mode) 비디오 분석 시작...

========================= DEBUGGING ANALYSIS =========================
⏱️ 추론 소요 시간: 6.11초
-------------------------------------------------------------------
🏁 종료 사유(Finish Reason): stop
📝 [최종 답변]:
[Activity Log]
- Interval 1 : A woman wearing a purple shirt and red patterned pants stands facing a kitchen counter, appearing to prepare food or wash items. She remains stationary at the counter, leaning slightly forward as she works with her hands.
- Interval 2 : The woman continues to stand at the kitchen counter, making small movements with her upper body and arms as she engages in her task. She shifts her weight slightly but maintains her position facing the counter.
- Interval 3 : She remains at the counter, continuing her activity. Her posture is upright but leans forward slightly towards the workspace. She adjusts her position minimally while keeping her focus on the counter area.
- Interval 4 : The woman stays in the same spot at the ki

In [15]:
# import os
# os.system("pkill -f api_server")
# print("🛑 기존 vLLM 서버 종료 완료!")

🛑 기존 vLLM 서버 종료 완료!
time: 6.61 ms (started: 2026-05-11 15:49:27 +00:00)


In [ ]:
# # <4>
# # 1. System Prompt
# system_prompt = """
# You are an objective video analysis AI. Your output must ONLY consist of the [Activity Log] section.

# [Strict Constraints]
# 1. Factual Description Only: Describe physical movements, velocity, and balance. If the center of gravity shifts faster than a controlled sitting motion, specify it as a 'loss of balance' or 'sudden collapse'.
# 2. No Summary/Status: Do not provide "Current Status", "Summary of Concerns", or "Movement Quality" sections.
# 3. Describe the direction of the fall based on which body part makes contact with the floor first (e.g., knees, chest, side, or back). Do not assume the direction unless the movement trajectory is clearly visible.
# 4. No Inference: Describe the fall as an "uncontrolled rapid descent" if it lacks bracing motions, but do not guess the medical cause.
# 5. Output Format: Strictly follow the format below without any additional text.
# """

# # 2. User Prompt
# user_prompt = f"""
# Analyze the provided video clip ({DURATION}s). 
# Provide a factual description of the movements for each 10-second interval.

# ### [Activity Log]
# - Interval 1 ({START_TIME}.0s - {START_TIME + 10}.0s): 
# - Interval 2 ({START_TIME + 10}.0s - {START_TIME + 20}.0s):
# """


### 3-3) 메시지 구성, 파라미터 출력, 분석, 결과 출력

In [ ]:
# # 3. 메시지 구성
# messages = [
#     {"role": "system", "content": system_prompt},
#     {
#         "role": "user",
#         "content": [
#             {"type": "video", "video": TEMP_CLIP, "fps": FPS}, 
#             {"type": "text", "text": user_prompt.strip()}
#         ],
#     }
# ]

# # 4. 파라미터 설정 (중요: 온도를 높여 두 번째 문장 생성을 유도)
# sampling_params = SamplingParams(
#     temperature=0.3,         # 0.01은 너무 딱딱해서 첫 줄만 쓰고 멈출 수 있음
#     max_tokens=512, 
#     # top_p=0.9,
#     repetition_penalty=1.2,  # 동일 문구 무한 반복 방지
#     # presence_penalty=1.1,   # 새로운 관찰 내용 유도
#     # stop=["###"]  # 리포트가 끝나면 정지
# )

# # 5. 분석 수행
# print(f"🚀 Qwen3-VL 모델 분석 시작... (분석 구간: {START_TIME}s ~ {START_TIME + DURATION}s)(VRAM 사용량 및 추론 시간을 확인하세요)")
# inputs = [prepare_inputs(messages, processor)]
# outputs = llm.generate(inputs, sampling_params=sampling_params)

# # 6. 결과 출력
# print("\n" + "="*25 + " ANALYSIS REPORT " + "="*25)
# print(f"📂 대상 파일: {os.path.basename(ORIGIN_VIDEO)}")
# print(f"🎥 분석 설정: {START_TIME}s 시점부터 {DURATION}초간 (Sampling: {FPS}fps)")
# print("-" * 67)
# print(outputs[0].outputs[0].text.strip())
# print("=" * 67)

In [8]:
# import os
# import glob
# import subprocess

# MODEL_PATH = "/workspace/models/Qwen3.6-35B-A3B-FP8"
# print("🔍 손상된 JSON 파일 검사 및 복구 시작...")

# # 1. 모델 폴더 내의 모든 json 파일 찾기
# json_files = glob.glob(os.path.join(MODEL_PATH, "*.json"))

# # 2. 내용이 비어있거나(0바이트) 깨진 파일 삭제
# deleted_files = 0
# for file_path in json_files:
#     # 파일 크기가 10바이트 이하인 경우 (정상적인 json은 최소 수백 바이트임)
#     if os.path.getsize(file_path) < 10:
#         print(f"🗑️ 손상된 파일 발견 및 삭제: {os.path.basename(file_path)}")
#         os.remove(file_path)
#         deleted_files += 1

# if deleted_files > 0:
#     print(f"\n총 {deleted_files}개의 깨진 파일을 지웠습니다. 누락된 파일을 다시 다운로드합니다 (수 초 소요)...")
#     # 3. hf download 명령어로 누락된 json 파일들만 쏙쏙 다시 채워넣기
#     cmd = f"hf download Qwen/Qwen3.6-35B-A3B-FP8 --local-dir {MODEL_PATH}"
#     subprocess.run(cmd, shell=True)
#     print("✅ 설정 파일 복구 완료! 이제 다시 서버 기동 셀을 실행해 주세요.")
# else:
#     print("❌ 0바이트 파일은 없습니다. 다른 파일이 깨졌을 수 있으니 모든 json을 다시 받습니다.")
#     os.system(f"rm {MODEL_PATH}/*.json")
#     os.system(f"hf download Qwen/Qwen3.6-35B-A3B-FP8 --local-dir {MODEL_PATH}")
#     print("✅ 모든 JSON 설정 파일 초기화 및 복구 완료!")

🔍 손상된 JSON 파일 검사 및 복구 시작...
❌ 0바이트 파일은 없습니다. 다른 파일이 깨졌을 수 있으니 모든 json을 다시 받습니다.
✅ 모든 JSON 설정 파일 초기화 및 복구 완료!
time: 53.9 ms (started: 2026-05-11 15:46:53 +00:00)


sh: 1: hf: not found


In [18]:
# print("🔍 누락된 설정 파일 복구 중...")

# try:
#     from huggingface_hub import snapshot_download
# except ImportError:
#     import os
#     os.system("pip install huggingface_hub")
#     from huggingface_hub import snapshot_download

# # 무거운 가중치 파일(*.safetensors)은 제외하고, 누락된 텍스트/json 파일만 즉시 채워 넣습니다.
# snapshot_download(
#     repo_id="Qwen/Qwen3.6-35B-A3B-FP8",
#     local_dir="/workspace/models/Qwen3.6-35B-A3B-FP8",
#     ignore_patterns=["*.safetensors"] 
# )

# print("✅ 파일 복구 완료! 이제 [Cell 2] (서버 기동) 셀을 다시 실행해 주세요.")

🔍 누락된 설정 파일 복구 중...


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

✅ 파일 복구 완료! 이제 [Cell 2] (서버 기동) 셀을 다시 실행해 주세요.
time: 3.83 s (started: 2026-05-11 15:54:07 +00:00)


In [ ]:
workspace/data/video/ai_hub_senior_anomaly/Validation/video/dementia_daily/M_I_004/DDA_In_MI004_00009.mp4